# 05 – Feature Engineering

**Objective:** turn each player's history into pre-match information a model can learn from, while making sure no future information leaks in.

**Key results**
- **Data:** 861,108 matches across all professional levels, used to build 61 features for 78,876 main-tour matches.
- **Better Elo:** adding lower-level matches, an inactivity penalty and a new-player boost lowered log loss from 0.6109 to 0.5971 on 2015–2021.
- **Leakage:** both tests pass, and a subtle leak found along the way was fixed.
- **Home advantage:** home players win 3.6 points more than Elo predicts, with a margin of error of ±0.9.

The logic lives in `src/`: `tennis_data.py` loads and orders matches, `tennis_elo.py` computes ratings, and `tennis_features.py` builds everything else.

In [1]:
import sqlite3
import sys
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from tennis_data import load_all_matches
from tennis_elo import combined_difference, log_loss_of_difference, run_elo, win_probability
from tennis_features import FEATURE_GROUPS, build_features

RAW_DIR = Path("../data/raw")
DB_PATH = Path("../data/processed/tennis.db")
BASE_URL = "https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/main/atp/"

## 1. More data: every level of professional tennis

Notebook 04 only saw the main tour, so a player arrived with no history at all. Here we add **qualifying, Challenger and Futures** matches, plus main-tour matches from the 1990s to warm up the ratings.

**The test period is still protected:** nothing from 2022 onwards is scored here or used to choose a setting.

In [2]:
names = (
    [f"atp_matches_{year}.csv" for year in range(1990, 2027)]
    + [f"atp_matches_qual_chall_{year}.csv" for year in range(1990, 2027)]
    + [f"atp_matches_futures_{year}.csv" for year in range(1991, 2027)]
    + ["atp_players.csv"]
)
for name in names:
    path = RAW_DIR / name
    if not path.exists():
        urllib.request.urlretrieve(BASE_URL + name, path)

matches = load_all_matches(RAW_DIR)
players = pd.read_csv(RAW_DIR / "atp_players.csv", low_memory=False)
tournament_info = pd.read_csv("../reference/tournament_info.csv")

print(f"{len(matches):,} matches")
matches["tier"].value_counts()

861,108 matches


tier
futures       523355
challenger    140869
main          116449
qualifying     80435
Name: count, dtype: int64

## 2. A better Elo

Every version is scored on the same matches: main-tour matches from 2015–2021, without Davis Cup or walkovers.

- **All levels:** Challenger and Futures results now count, so newcomers arrive with a meaningful rating.
- **Inactivity:** after 60 days away, a rating moves back towards 1500. After a full extra year away, it loses 30% of its distance from 1500.
- **New-player boost:** ratings move faster for players with few matches, so they reach their true level sooner.

For each version, the table shows the best weight between overall and surface ratings.

In [3]:
validation = (
    (matches["source"] == "main") & matches["year"].between(2015, 2021)
    & (matches["tourney_level"] != "D") & ~matches["walkover"]
)
WEIGHTS = [1, 0.75, 0.6, 0.5, 0.4, 0.25]


def evaluate_elo(label, data, **settings):
    ratings = run_elo(data, **settings)
    mask = validation.loc[data.index]
    loss, weight = min(
        (log_loss_of_difference(combined_difference(ratings[mask], w)), w) for w in WEIGHTS
    )
    accuracy = (combined_difference(ratings[mask], weight) > 0).mean()
    return {"version": label, "overall_weight": weight, "accuracy_pct": round(100 * accuracy, 2), "log_loss": round(loss, 4)}


main_only = matches[(matches["source"] == "main") & (matches["year"] >= 2000)]
pd.DataFrame([
    evaluate_elo("Main tour only (notebook 04 method)", main_only, k=32),
    evaluate_elo("All levels, K = 32", matches, k=32),
    evaluate_elo("All levels, K = 28", matches, k=28),
    evaluate_elo("+ inactivity", matches, k=28, inactivity=0.3),
    evaluate_elo("+ inactivity and new-player boost", matches, k=28, inactivity=0.3, new_player_boost=1.0),
])

,version,overall_weight,accuracy_pct,log_loss
0,Main tour only (notebook 04 method),0.6,66.15,0.6109
1,"All levels, K = 32",0.6,67.35,0.6003
2,"All levels, K = 28",0.6,67.43,0.6001
3,+ inactivity,0.6,67.52,0.5975
4,+ inactivity and new-player boost,0.6,67.56,0.5971


**Finding:** adding lower-level matches gives the largest gain by far: accuracy rises from 66.15% to 67.35%. Lower-level results tell us a lot about players who are new to the main tour, whom a main-tour-only Elo barely knows. These settings were chosen on this same period, so the figures are slightly optimistic.

## 3. Build all the features

The code walks through all 861,108 matches in order. For each match it **first reads** each player's history, and **then updates** that history with the result.

Each feature is "winner minus loser". Because the model is trained on both points of view (see notebook 06), it can't use the fact that the winner is listed first.

In [4]:
ELO_SETTINGS = dict(k=28, inactivity=0.3, new_player_boost=1.0)
ELO_WEIGHT = 0.6

ratings = run_elo(matches, **ELO_SETTINGS)
features = build_features(matches, ratings, players, tournament_info, ELO_WEIGHT)
print(f"{len(features):,} matches, {features.shape[1]} columns")

pd.DataFrame(
    [{"group": group, "n_features": len(columns), "examples": ", ".join(columns[:3])}
     for group, columns in FEATURE_GROUPS.items()]
)

78,876 matches, 67 columns


,group,n_features,examples
0,Elo,3,"d_elo_combined, d_elo_overall, d_elo_surface"
1,Ranking,3,"d_log_rank, d_log_points, d_unranked"
2,Serve and return,13,"d_serve_fast, d_serve_slow, d_return_fast"
3,Form,6,"d_win_last_5, d_win_last_10, d_win_last_20"
4,Fatigue and fitness,9,"d_matches_7, d_matches_14, d_matches_30"
5,Experience,3,"d_log_career, d_log_tour_career, d_log_surface..."
6,Style and age,4,"d_height, d_age, d_lefty"
7,Head to head,3,"d_h2h_wins, d_h2h_share, d_h2h_surface"
8,Draw,5,"d_seed, d_qualifier, d_wildcard"
9,Home,1,d_home


## 4. Testing for data leakage

**Test 1: remove the future.** If features for 2018 matches are identical whether or not later years exist, they aren't using the future.

In [5]:
FEATURE_COLUMNS = [column for columns in FEATURE_GROUPS.values() for column in columns]

past = matches[matches["year"] <= 2018]
past_features = build_features(past, run_elo(past, **ELO_SETTINGS), players, tournament_info, ELO_WEIGHT)
full_features = features[features["year"] <= 2018].reset_index(drop=True)

difference = np.abs(past_features[FEATURE_COLUMNS].fillna(0).to_numpy() - full_features[FEATURE_COLUMNS].fillna(0).to_numpy())
print(f"Compared {len(past_features):,} matches. Largest difference: {difference.max()}")

Compared 58,725 matches. Largest difference: 0.0


**Test 2: swap the result.** If one match's result is reversed, that match's own features must simply change sign. Any other change would mean they depend on the match's result.

The test uses matches where the two players had already met.

In [6]:
SWAPS = [("winner_id", "loser_id"), ("winner_seed", "loser_seed"), ("winner_entry", "loser_entry"),
         ("winner_age", "loser_age"), ("winner_rank", "loser_rank"), ("winner_rank_points", "loser_rank_points")]
SWAPS += [(f"w_{s}", f"l_{s}") for s in ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "bpSaved", "bpFaced"]]

recent = matches[matches["year"] >= 2016]
base = build_features(recent, run_elo(recent, **ELO_SETTINGS), players, tournament_info, ELO_WEIGHT)
direction_columns = [c for c in FEATURE_COLUMNS if c.startswith("d_")]

candidates = base[(base["year"] == 2019) & (base["d_h2h_share"] != 0)].sample(5, random_state=1)
largest = 0.0
for match in candidates.itertuples():
    row = recent[(recent["source"] == "main") & (recent["tourney_id"] == match.tourney_id) & (recent["match_num"] == match.match_num)].index
    swapped = recent.copy()
    for a, b in SWAPS:
        swapped.loc[row, [a, b]] = swapped.loc[row, [b, a]].to_numpy()
    rebuilt = build_features(swapped, run_elo(swapped, **ELO_SETTINGS), players, tournament_info, ELO_WEIGHT)
    original = base.loc[match.Index, direction_columns].astype(float).fillna(0).to_numpy()
    reversed_view = rebuilt.loc[match.Index, direction_columns].astype(float).fillna(0).to_numpy()
    largest = max(largest, np.abs(original + reversed_view).max())

print(f"Largest gap between a match's features and their exact opposite: {largest}")

Largest gap between a match's features and their exact opposite: 0.0


### A leak that these tests didn't catch at first

In an early version, gradient boosting suddenly reached **73% accuracy with a log loss of 0.509**, far above every other result. That looked too good, so it was investigated rather than celebrated.

Removing one group at a time pointed straight at head to head, yet its values were all computed from past matches. The cause was a **rounding effect**:

```python
3/7 - 0.5  # -0.07142857142857145
4/7 - 0.5  #  0.07142857142857140
```

The two values should be exact opposites, but the last digits differ. Flipping the sign of a row therefore left a tiny fingerprint showing which view was the original, meaning the winner's. The boosting model learned to read that fingerprint.

**The fix:** the feature is now written as one difference, and all features are rounded to 8 decimals. Accuracy went back to about 68%. Test 2 above now uses matches where the players had already met, so it would catch this.

**The lesson:** a result that looks too good is a bug until proven otherwise.

## 5. Home advantage, measured properly

Does a home player win more than their Elo predicts? This uses matches with exactly one home player, from 2005–2021.

In [7]:
names = matches[matches["source"] == "main"][["tourney_id", "match_num", "tourney_name"]]
data = features.merge(names, on=["tourney_id", "match_num"])
home = data[(data["one_home_player"] == 1) & data["year"].between(2005, 2021) & (data["tourney_level"] != "D")].copy()

# Everything from the home player's point of view
home_won = home["d_home"] > 0
elo_for_winner = win_probability(home["d_elo_combined"], 0)
home["expected"] = np.where(home_won, elo_for_winner, 1 - elo_for_winner)
home["won"] = home_won.astype(float)
home["event"] = home["tourney_name"].replace({"Us Open": "US Open"})
home.loc[home["tourney_level"] != "G", "event"] = "All other tournaments"


def summarise(group):
    gap = group["won"].mean() - group["expected"].mean()
    margin = 1.96 * np.sqrt((group["expected"] * (1 - group["expected"])).sum()) / len(group)
    return pd.Series({"n_matches": len(group), "actual_pct": 100 * group["won"].mean(),
                      "elo_expected_pct": 100 * group["expected"].mean(),
                      "gap_points": 100 * gap, "margin_of_error": 100 * margin})


table = home.groupby("event")[["won", "expected"]].apply(summarise).round(1)
table.loc["All matches"] = summarise(home).round(1)
table

,n_matches,actual_pct,elo_expected_pct,gap_points,margin_of_error
event,,,,,
All other tournaments,8789.0,48.5,44.9,3.7,0.9
Australian Open,249.0,41.4,36.4,5.0,5.2
Roland Garros,503.0,46.3,43.1,3.3,3.8
US Open,473.0,45.5,43.4,2.1,3.9
Wimbledon,169.0,50.9,45.1,5.8,6.1
All matches,10183.0,48.1,44.5,3.6,0.9


**Finding:** the gap of 3.6 points is well above its margin of error of ±0.9, so a home effect is very likely real. Individual Grand Slams rest on far fewer matches, so their gaps are much less certain. This doesn't prove the cause, which could be crowds, travel, familiar conditions or wild cards going to players who are better than their record suggests. Whether it improves predictions is tested in notebook 06.

## 6. Save the features

In [8]:
conn = sqlite3.connect(DB_PATH)
features.to_sql("match_features", conn, if_exists="replace", index=False)
tournament_info.to_sql("tournament_info", conn, if_exists="replace", index=False)
conn.commit()

pd.read_sql_query("SELECT COUNT(*) AS rows_saved FROM match_features", conn)

,rows_saved
0,78876
